# 环节 03 · 文件系统隔离与快照演示

纯 Python 标准库，零依赖。手搓四件事：

1. overlayfs 三层合并（lower / upper / whiteout）；
2. copy-up 成本放大；
3. 保护路径判定（可写目录内的执行入口）；
4. 快照恢复 vs 冷启动的时间账。

In [ ]:
# §1 overlayfs 合并视图模拟
class Overlay:
    def __init__(self, lower, upper=None):
        self.lower = dict(lower)          # 只读层
        self.upper = dict(upper or {})    # 可写层
        self.whiteout = set()             # 删除标记（whiteout）

    def read(self, path):
        if path in self.whiteout:
            return "(deleted: whiteout)"
        if path in self.upper:
            return self.upper[path]
        return self.lower.get(path, "(no such file)")

    def write(self, path, content):
        cost = len(self.lower.get(path, "")) if path not in self.upper else 0
        self.upper[path] = content
        return cost                       # 返回 copy-up 的代价

    def delete(self, path):
        self.upper.pop(path, None)
        self.whiteout.add(path)

    def merged(self):
        names = (set(self.lower) | set(self.upper)) - self.whiteout
        return sorted(names)


ov = Overlay({"/usr/bin/python": "bin", "/etc/app.conf": "v1", "/data/keep": "x"})
print("合并视图:", ov.merged())
print("读 /etc/app.conf      →", ov.read("/etc/app.conf"))

cost = ov.write("/etc/app.conf", "v2-larger-content")
print(f"写 /etc/app.conf      → 发生 copy-up，代价 = {cost} 字节（整份复制）")
print("读 /etc/app.conf      →", ov.read("/etc/app.conf"))

ov.delete("/data/keep")
print("删除 /data/keep 后视图:", ov.merged())
print("读 /data/keep         →", ov.read("/data/keep"))

In [ ]:
# §2 copy-up 放大：pip install 的隐性成本
FILES = 80000
AVG_KB = 12
copy_up_bytes = FILES * AVG_KB * 1024

print(f"沙箱内 pip install：{FILES:,} 个新文件 / 平均 {AVG_KB} KB")
print(f"upper 层写入量 ≈ {copy_up_bytes / 1024 ** 3:.2f} GB")
print()
print("若 upper 落在 tmpfs（内存盘）：这 0.9 GB 直接吃内存")
print("→ 典型事故：'沙箱内存莫名涨'，根因是 upper 用了 tmpfs")

In [ ]:
# §3 可写目录内的"执行入口"必须单独保护
import fnmatch

PROTECTED = [
    "**/.bashrc", "**/.zshrc", "**/.gitconfig",
    "**/.git/config", "**/.git/hooks/*",
    "**/.env", "**/.ssh/*", "**/.aws/credentials",
    "**/.claude/*", "**/.mcp.json", "**/.vscode/*", "**/.idea/*",
]

def is_protected(path, patterns=PROTECTED):
    return any(fnmatch.fnmatch(path, p) for p in patterns)


TESTS = [
    "/work/src/main.py", "/work/.bashrc", "/work/.git/config",
    "/work/.git/hooks/pre-commit", "/work/.env", "/work/.vscode/settings.json",
    "/home/u/.ssh/id_rsa", "/work/README.md",
]
for p in TESTS:
    print(f"{'拒绝写' if is_protected(p) else '允许写'}  {p}")
print()
print("原则：可写目录 ≠ 整棵子树可写。凡是『下次运行会被执行/加载』的都要单独拉黑")

In [ ]:
# §4 快照恢复 vs 冷启动
COLD = {"调度": 120, "镜像拉取": 0, "内核 boot": 150, "OS/运行时": 200, "应用初始化": 900}
SNAP = {"调度": 120, "镜像拉取": 0, "内核 boot": 0,   "OS/运行时": 40,  "应用初始化": 60}

def total(d):
    return sum(d.values())


print(f"{'阶段':<12}{'冷启动(ms)':<12}{'快照恢复(ms)'}")
print("-" * 38)
for k in COLD:
    print(f"{k:<12}{COLD[k]:<12}{SNAP[k]}")
print("-" * 38)
print(f"{'合计':<12}{total(COLD):<12}{total(SNAP)}")
saved = 1 - total(SNAP) / total(COLD)
print(f"\n快照可省 ≈ {saved:.0%} 的冷启动时间")
print("→ 判断标准：应用初始化占比高才值得上快照（占比低先优化调度/镜像）")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | copy-up 的代价？ | 首次修改要复制**整个文件**，大量小文件写入会被放大 |
| 2 | whiteout 是什么？ | upper 层里表示"下层文件已删除"的标记 |
| 3 | 为什么"可写目录"还不够？ | 目录内有执行入口（shell rc、hooks、git config） |
| 4 | 读和写哪个更危险？ | 读（凭证外泄配合网络 = 完整泄露链） |
| 5 | upper 层放 tmpfs 的风险？ | 写入量直接吃内存，是"内存莫名涨"的常见根因 |
| 6 | 快照里最不该有什么？ | 凭证（会随快照复制传播） |

**相关长文**：[环节03-文件系统隔离与快照详解.md](./环节03-文件系统隔离与快照详解.md)